In [68]:
from langgraph.graph import StateGraph ,START,END
from typing import TypedDict


In [69]:
class BatsmanState(TypedDict):
    
    runs:int
    balls:int
    fours:int
    sixes:int

    sr:float
    bpb:float
    boundery_percent:float
    summary:str


In [70]:
def calculate_sr(State:BatsmanState):

    sr=(State['runs']/State['balls'])*100

    State['sr']=sr
    
    return {'sr':sr}



In [71]:
def calculate_bpb(state:BatsmanState):

    bpb=state['balls']/(state['fours']+state['sixes'])

    state['bpb']=bpb
    return {'bpb':bpb}

In [72]:
def calculate_boundary_percent(state:BatsmanState):

    boundery_percent=(((state['fours']*4)+(state['sixes']*6))/state['runs'])*100

    state['boundery_percent']=boundery_percent

    return {'boundery_percent':boundery_percent}



In [73]:
def summary(state:BatsmanState):
    summary=f"""
        Strike_rate -{state['sr']}\n
        balls per   -{state['bpb']} \n
        Boundery percent - {state['boundery_percent']}
        """
    state['summary']=summary

    return {'summary':summary}

In [74]:
graph = StateGraph(BatsmanState)

# 1. Add Nodes
graph.add_node('calculate_str', calculate_sr) 
graph.add_node('calculate_bpb', calculate_bpb) 
# Notice the "a" in boundary here:
graph.add_node('calculate_boundary_percent', calculate_boundary_percent) 
graph.add_node('summary', summary)

# 2. Add Edges from START
graph.add_edge(START, 'calculate_str')
graph.add_edge(START, 'calculate_bpb')
# FIXED: Changed 'boundery' to 'boundary'
graph.add_edge(START, 'calculate_boundary_percent')

# 3. Add Edges to summary
graph.add_edge('calculate_str', 'summary')
graph.add_edge('calculate_bpb', 'summary')
# FIXED: Changed 'boundery' to 'boundary'
graph.add_edge('calculate_boundary_percent', 'summary')

# 4. Add End Edge
graph.add_edge('summary', END)

# 5. Compile
workflow = graph.compile()
print(workflow)


In [77]:
initial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

output=workflow.invoke(intial_state)

In [78]:
print(output['summary'])


        Strike_rate -200.0

        balls per   -5.0 

        Boundery percent - 48.0
        
